# Cardiac Patient Monitoring System

## 03 — Supervised Learning

## Objective

Define the supervised classification problem, create a reproducible stratified train/test split, establish a Logistic Regression baseline, and compare it with a second classifier.

## Imports and Load Data

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

DATA_PATH = DATA_DIR / "cardio_train.csv"

OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

CLEAN_PATH = DATA_DIR / "cardio_clean.csv"
if not CLEAN_PATH.exists():
    raise FileNotFoundError("Run 01_data_preparation.ipynb first.")
df = pd.read_csv(CLEAN_PATH)


## Define Features and Target

In [ ]:
TARGET = "cardio"
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("Train target distribution:")
display(y_train.value_counts(normalize=True).sort_index())
print("Test target distribution:")
display(y_test.value_counts(normalize=True).sort_index())


## Baseline — Logistic Regression

In [ ]:
baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
baseline_prob = baseline.predict_proba(X_test)[:, 1]

baseline_metrics = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, baseline_pred),
    "Precision": precision_score(y_test, baseline_pred),
    "Recall": recall_score(y_test, baseline_pred),
    "F1": f1_score(y_test, baseline_pred),
    "ROC-AUC": roc_auc_score(y_test, baseline_prob)
}
pd.DataFrame([baseline_metrics])


## Comparison Model — Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight=None
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

rf_metrics = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_test, rf_pred),
    "Precision": precision_score(y_test, rf_pred),
    "Recall": recall_score(y_test, rf_pred),
    "F1": f1_score(y_test, rf_pred),
    "ROC-AUC": roc_auc_score(y_test, rf_prob)
}
pd.DataFrame([baseline_metrics, rf_metrics])


## Compare the Models

In [ ]:
results = pd.DataFrame([baseline_metrics, rf_metrics]).set_index("Model")
display(results.round(4))

results.to_csv(OUTPUT_DIR / "initial_model_comparison.csv")


## Why These Models?

- **Logistic Regression** provides a simple, interpretable linear baseline.
- **Random Forest** provides a non-linear tree-based comparison model.
- Both models are evaluated on the same train/test split so the comparison is consistent.

## Initial Conclusion

The stronger model should not be selected from accuracy alone. The next notebook evaluates both models using cross-validation and a full classification report, confusion matrix, and ROC-AUC.